# 05 Evaluation

Notebook untuk menampilkan metrik evaluasi retrieval dan prediksi solusi.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

PROJECT_ROOT = Path('..').resolve()
DATA_DIR = PROJECT_ROOT / 'data'
RAW_PDF_DIR = DATA_DIR / 'raw' / 'pdf'
RAW_TEXT_DIR = DATA_DIR / 'raw' / 'text'
PROCESSED_DIR = DATA_DIR / 'processed'
EVAL_DIR = DATA_DIR / 'eval'
RESULTS_DIR = DATA_DIR / 'results'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
retrieval_metrics = pd.read_csv(EVAL_DIR / 'retrieval_metrics.csv')
prediction_metrics = pd.read_csv(EVAL_DIR / 'prediction_metrics.csv')
print('Retrieval Metrics')
display(retrieval_metrics)
print('Prediction Metrics')
display(prediction_metrics)

In [ ]:
def save_metric_bar(metrics_df, output_path, title, color):
    plot_df = metrics_df.copy()
    plot_df['value'] = pd.to_numeric(plot_df['value'], errors='coerce')

    fig, ax = plt.subplots(figsize=(9, 4.8))
    bars = ax.bar(plot_df['metric'], plot_df['value'], color=color)
    ax.set_ylim(0, 1)
    ax.set_ylabel('Nilai')
    ax.set_title(title)
    ax.tick_params(axis='x', rotation=25)
    ax.grid(axis='y', linestyle='--', alpha=0.35)

    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width() / 2, height + 0.02, f'{height:.2f}', ha='center', va='bottom', fontsize=9)

    fig.tight_layout()
    fig.savefig(output_path, dpi=160)
    plt.close(fig)


def save_combined_metrics(retrieval_df, prediction_df, output_path):
    combined = pd.concat([
        retrieval_df.assign(kategori='Retrieval'),
        prediction_df.assign(kategori='Prediction'),
    ], ignore_index=True)
    combined['label'] = combined['kategori'] + ': ' + combined['metric']
    combined['value'] = pd.to_numeric(combined['value'], errors='coerce')
    colors = combined['kategori'].map({'Retrieval': '#2f6f9f', 'Prediction': '#5b8c5a'}).fillna('#777777')

    fig, ax = plt.subplots(figsize=(11, 5.8))
    bars = ax.barh(combined['label'], combined['value'], color=colors)
    ax.set_xlim(0, 1)
    ax.set_xlabel('Nilai')
    ax.set_title('Ringkasan Metrik Retrieval dan Prediksi')
    ax.grid(axis='x', linestyle='--', alpha=0.35)

    for bar in bars:
        width = bar.get_width()
        ax.text(width + 0.015, bar.get_y() + bar.get_height() / 2, f'{width:.2f}', va='center', fontsize=9)

    fig.tight_layout()
    fig.savefig(output_path, dpi=160)
    plt.close(fig)


def save_confusion_matrix_heatmap(confusion_df, output_path):
    matrix = confusion_df.set_index(confusion_df.columns[0])
    fig, ax = plt.subplots(figsize=(8.5, 6.5))
    image = ax.imshow(matrix.values, cmap='Blues')
    ax.set_xticks(range(len(matrix.columns)))
    ax.set_yticks(range(len(matrix.index)))
    ax.set_xticklabels(matrix.columns, rotation=35, ha='right')
    ax.set_yticklabels(matrix.index)
    ax.set_xlabel('Prediksi')
    ax.set_ylabel('Aktual')
    ax.set_title('Confusion Matrix Prediksi Solusi')

    max_value = matrix.values.max() if matrix.size else 0
    for i in range(matrix.shape[0]):
        for j in range(matrix.shape[1]):
            value = matrix.iat[i, j]
            color = 'white' if max_value and value > max_value / 2 else 'black'
            ax.text(j, i, str(value), ha='center', va='center', color=color)

    fig.colorbar(image, ax=ax, fraction=0.046, pad=0.04)
    fig.tight_layout()
    fig.savefig(output_path, dpi=160)
    plt.close(fig)


save_metric_bar(retrieval_metrics, RESULTS_DIR / 'retrieval_metrics_bar.png', 'Metrik Retrieval Top-5', '#2f6f9f')
save_metric_bar(prediction_metrics, RESULTS_DIR / 'prediction_metrics_bar.png', 'Metrik Prediksi Solusi', '#5b8c5a')
save_combined_metrics(retrieval_metrics, prediction_metrics, RESULTS_DIR / 'combined_metrics_bar.png')
save_confusion_matrix_heatmap(pd.read_csv(EVAL_DIR / 'confusion_matrix.csv'), RESULTS_DIR / 'confusion_matrix_heatmap.png')
print('Artefak evaluasi berhasil dibuat di:', RESULTS_DIR)


In [ ]:
pd.read_csv(EVAL_DIR / 'retrieval_eval_detail.csv').head()

In [ ]:
pd.read_csv(EVAL_DIR / 'prediction_eval_detail.csv')

In [ ]:
failure = pd.read_csv(EVAL_DIR / 'failure_analysis.csv')
print('Jumlah failure case:', len(failure))
failure

## Visualisasi Tambahan

Bagian ini menampilkan grafik evaluasi yang sudah disimpan di folder `data/results/`.

In [ ]:
from IPython.display import Image, display
for name in ['retrieval_metrics_bar.png','prediction_metrics_bar.png','combined_metrics_bar.png','confusion_matrix_heatmap.png']:
    print(name)
    display(Image(filename=str(RESULTS_DIR / name)))